In [1]:
from load_data import domain_var, load_data, temperature_var, velocity_var, x_var

import qewton

In [2]:
dataset, data_loader = load_data(5)
print(domain_var)
print(temperature_var)
print(velocity_var)
print(len(dataset))

Variable(domain, dim=1)
Variable(temperature, dim=1)
Variable(velocity, dim=3, children=3)
581


In [3]:
def visualize_mesh(index, variable=temperature_var, mode="slices", n_slices=8, color=True):
    data = dataset.load_complete_data(variable=variable)
    data_config = next(c for c in dataset.data_configs if variable in c.variables)
    index_spec = qewton.viz.FixedSpec(
        init_state=index, variable_or_axes=data_config.axes[0]
    )
    is_vector = False
    if variable.shape is not None and isinstance(variable.shape[0], int):
        is_vector = variable.shape[0] > 1 or (isinstance(variable.dim, tuple) and len(variable.dim) > 1)

    if mode == "volume":
        plot = (
            qewton.viz.QuiverPlot(
                data,
                data_config,
                vector=qewton.viz.VectorSpec(variable),
                controls=[index_spec],
            )
            if is_vector
            else qewton.viz.PointCloudPlot(
                data,
                data_config,
                color=qewton.viz.ColorSpec(variable),
                controls=[index_spec],
            )
        )
    else:
        geometry = data_config.geometry_axes.geometry
        n_layers = geometry.shape[0]
        grid_axis = geometry.variable.leaves[0]
        if n_slices > 1:
            layers = [i * (n_layers - 1) // (n_slices - 1) for i in range(n_slices)]
        else:
            layers = [n_layers // 2]
        scale = qewton.viz.Scale() if color else qewton.viz.Scale(0, 0)
        plots = []
        for layer_idx in layers:
            controls = [
                index_spec,
                qewton.viz.FixedSpec(init_state=layer_idx, variable_or_axes=grid_axis),
            ]
            plots.append(
                qewton.viz.QuiverPlot(
                    data,
                    data_config,
                    vector=qewton.viz.VectorSpec(variable, color_by_magnitude=color),
                    controls=controls,
                )
                if is_vector
                else qewton.viz.EmbeddedGridPlot(
                    data,
                    data_config,
                    color=qewton.viz.ColorSpec(variable, scale=scale),
                    controls=controls,
                )
            )
        plot = qewton.viz.Overlay(*plots)

    figure = qewton.viz.Figure(
        plot, title=f"{variable.name} (sample {index})",
        theme=qewton.viz.Theme(opacity_default=1)
    )
    figure.show()

In [4]:
# the evenly spaced domain coords
visualize_mesh(5, x_var, color=False)

In [5]:
# the domain mask itself
visualize_mesh(5, domain_var)

In [6]:
# temperature on slices
visualize_mesh(5)


In [7]:
# temperature as a point cloud
visualize_mesh(5, mode="volume")

In [8]:
# velocity arrows on the layer stack
visualize_mesh(5, velocity_var)